# NB_lab_test_trends

Builds the **gold.lab_test_trends** table — one row per test type per calendar month with abnormal/critical rates and month-over-month trends. Source: STTM Silver → Gold.

In [0]:
%sql
CREATE OR REPLACE TABLE databricks_worksoace.gold.lab_test_trends
USING DELTA
AS
WITH monthly_agg AS (
  SELECT
    lr.test_name,
    YEAR(lr.test_date) AS year,
    MONTH(lr.test_date) AS month,
    COUNT(*) AS total_tests,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END) AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END) AS critical_test_count
  FROM databricks_worksoace.silver.lab_results lr
  GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
),
with_rates AS (
  SELECT
    ma.test_name,
    ma.year,
    ma.month,
    c.month_name,
    c.quarter_name,
    ma.total_tests,
    ma.abnormal_test_count,
    ma.critical_test_count,
    CAST(ma.abnormal_test_count / ma.total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ma.critical_test_count / ma.total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly_agg ma
  LEFT JOIN databricks_worksoace.default.calender c
    ON c.date_key = CAST(date_format(make_date(ma.year, ma.month, 1), 'yyyyMMdd') AS INT)
)
SELECT
  test_name,
  year,
  month,
  month_name,
  quarter_name,
  total_tests,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct,
  CAST(
    abnormal_rate_pct - LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month)
    AS DECIMAL(6,2)
  ) AS abnormal_rate_change_pct
FROM with_rates;

In [0]:
record_count = spark.table("databricks_worksoace.gold.lab_test_trends").count()
dbutils.notebook.exit(str(record_count))